# Demo 5.1: KV transfer one level down: the sub-sub-goal

**Question.** Demo 5 showed that the goal heads re-read the *sub-goal* from the task span: swapping their keys /
values over the task list redirects the model to another sub-task. Does the same motif work **one level down the
goal hierarchy**? Here the sub-goal is a filter with two conditions, and the contrast is between the two prompts'
*second* condition (the sub-sub-goal), which the model is about to write right after the connective.

**Setup.** `a'(x'z)` vs `a'(x'y)`. Both prompts share sub-task (1) `a` (prefilled) and the first condition `x` of the
filter in sub-task (2) (prefilled too). They differ only in the second condition: `z` in SRC, `y` in DST. The prefill
stops right after the connective (` and` in Python, ` &&` in C++): that token is the **pivot**.

```
p_src:  (1) a  (2) the list of words that x and z     ->  b = [w for w in words if <x> and  <z>
p_dst:  (1) a  (2) the list of words that x and y     ->  b = [w for w in words if <x> and  <y>
```

**Intervention.** Exactly Demo 5's: from the pivot on, the **250 goal heads** read SRC's keys / values over the task
span (values × 4); every other head and position is untouched. If DST now writes `z`, the goal heads carried the
sub-sub-goal over from the task span. Control: the same transfer through 250 random heads.

**Paper.** App. `app:task_transfer-hier` (`fig:kv-multi`, the sub-sub-goal panel). Pooled over 32 cells
({list[int], list[str]} × {and, or} × {match, diff} × 4 language pairs, n = 1024): causal shift ≈ 79%, readout shift
≈ 65%, random-head control 0 / 1024.

Needs one 80 GB GPU (Qwen2.5-Coder-32B-Instruct in bf16). The executed outputs are saved in this file.

**The setup** (the paper's figure; left: one instance, right: the pooled rates):

![Sub-sub-goal KV transfer: the C++ source's second condition (ends with a digit) is transplanted into a Python destination that would check for palindromes (App. I.1.1).](figures/05.1_kv_sub_sub_goal.png)

*Sub-sub-goal KV transfer (App. I.1.1): SRC's second filter condition is transplanted through the goal heads' view of
the task span, across languages.*

In [1]:
# repo root = first parent dir that contains src/
import os, sys
_d = os.getcwd()
while _d != '/' and not os.path.isdir(os.path.join(_d, 'src')):
    _d = os.path.dirname(_d)
sys.path.insert(0, _d); os.chdir(_d)
print('repo root:', _d)

# no progress bars: keeps tqdm / HF widget outputs out of the saved notebook
os.environ['TQDM_DISABLE'] = '1'
import functools, tqdm.std, tqdm.notebook, transformers, huggingface_hub
for _c in (tqdm.std.tqdm, tqdm.notebook.tqdm):   # tqdm.notebook ignores TQDM_DISABLE, so force disable=True
    _c.__init__ = functools.partialmethod(_c.__init__, disable=True)
transformers.utils.logging.disable_progress_bar()
huggingface_hub.utils.disable_progress_bars()

repo root: /disk/u/arnab/Codes/Projects/goals


## Config
Defaults are those of `scripts/nrgs/kv_transfer_eval.py --setup hier` (the job list
`job_list/diff_eval/path_patch_hier_32b_listgap.json`): task span, the 250 goal heads, V × 4, instances from
`sample_hier_instances(seed=1123)`, greedy 64 tokens, C++ library note in the system prompt.

**Why this instance.** `list[str]`, connective `and`, order `match`, **C++ → Python** (the transfer crosses
languages). Shared: (1) `any_empty`, first condition `x` = *short words*. SRC's second condition `z` = *starts with
'a'*; DST's `y` = *ends with a digit*. The two checks share nothing (`w.startswith('a')` vs `w[-1].isdigit()`), so a
flip is unambiguous, and for a Python destination it can also be checked by **executing** the completed
comprehension (`H.rule_verdict`). In the reported run this instance is a causal HIT with a valid, shifted readout
under G250 and a miss under the random-250 control.

In [2]:
CUDA     = '6'
MODEL    = 'Qwen/Qwen2.5-Coder-32B-Instruct'
G250     = 'head_sets/Qwen2.5-Coder-32B-Instruct/goal_topk/goal__lam20_top250.json'
C250     = 'head_sets/Qwen2.5-Coder-32B-Instruct/control/control250__laterhalf.json'
DATATYPE, CONNECTIVE, ORDER = 'list[str]', 'and', 'match'
SRC_LANG, DST_LANG = 'cpp', 'python'
INST_KEY = 'hier|list[str]|and|match|a=any_empty|x=short|y=last_is_digit|z=starts_a'   # (py destination: no language suffix)
SPAN     = 'task'      # the transplanted window: (1) .. the user turn's <|im_end|>
V_SCALE  = 4.0         # x the transplanted values read by the goal heads
MAX_NEW  = 64

os.environ['CUDA_VISIBLE_DEVICES'] = CUDA

## Load the model
`src.nrgs.functional.setup` loads the model (bf16, eager) and returns the run context the NRGS helpers take.

In [3]:
import json, torch
from rich.console import Console
from rich.table import Table
from rich.text import Text
from rich.markup import escape

import src.nrgs.data as DATA
import src.nrgs.functional as FUNC
import src.nrgs.hier_data as H
import src.nrgs.kv_transfer as PP
from src.utils import env_utils

console = Console(width=140, highlight=False)
torch.set_grad_enabled(False)
DATA.CPP_NOTE_IN_SYSTEM = True                     # as in kv_transfer_eval.main()

ctx = FUNC.setup(MODEL, device_map={'': 0})
load = lambda p: [tuple(h) for h in json.load(open(p))]
goal_heads, ctrl_heads = load(G250), load(C250)
ctx.readout_heads, ctx.gsp = goal_heads, None
print(f'goal heads: {len(goal_heads)} | control heads: {len(ctrl_heads)} (overlap {len(set(goal_heads) & set(ctrl_heads))})')

2026-11-03 00:24:03 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Qwen/Qwen2.5-Coder-32B-Instruct
n_heads=40  n_kv_heads=8  group=5  head_dim=128  n_layer=64
goal heads: 250 | control heads: 250 (overlap 0)


## 1. The contrastive pair
`sample_hier_instances` reproduces the eval's draw for this (datatype, connective, order, language pair); we pick our
instance by key. The instance knows its conditions: `x` (shared, prefilled), `src_cond` = `z` and `dst_cond` = `y`.
`build_geometry` is the same as in Demo 5: both prompts, the task span located in each, `\n`-padded to equal length.

In [4]:
insts, sample_seed = PP.sample_hier_instances(DATATYPE, CONNECTIVE, ORDER, SRC_LANG, DST_LANG, n_samples=32, seed=1123)
inst = next(i for i in insts if PP.inst_key(i) == INST_KEY)
g = PP.build_geometry(inst, span=SPAN, ctx=ctx)

t = Table(title=f'{escape(INST_KEY)}  (sample seed {sample_seed})')
t.add_column('role'); t.add_column(f'SRC ({SRC_LANG})'); t.add_column(f'DST ({DST_LANG})')
t.add_row('(1) shared sub-task a, prefilled', escape(inst.shared[0].description), escape(inst.shared[0].description))
t.add_row('(2) first condition x, prefilled', escape(inst.x.description), escape(inst.x.description))
t.add_row('(2) second condition (the sub-sub-goal)', f'[bold cyan]z: {escape(inst.src_cond.description)}[/]',
          f'[bold magenta]y: {escape(inst.dst_cond.description)}[/]')
console.print(t)
print(f'window: {len(g["dst_w"])} tokens, SRC [{g["src_w"][0]}..{g["src_w"][-1]}] / DST [{g["dst_w"][0]}..{g["dst_w"][-1]}]; '
      f'padding: SRC +{g["pad_src"]} / DST +{g["pad_dst"]} newline tokens')

        hier|list[str]|and|match|a=any_empty|x=short|y=last_is_digit|z=starts_a  (sample seed 3947442194)        
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ role                                    ┃ SRC (cpp)                        ┃ DST (python)                     ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ (1) shared sub-task a, prefilled        │ whether any word is empty        │ whether any word is empty        │
│ (2) first condition x, prefilled        │ are shorter than four characters │ are shorter than four characters │
│ (2) second condition (the sub-sub-goal) │ z: start with the letter a       │ y: end with a digit              │
└─────────────────────────────────────────┴──────────────────────────────────┴──────────────────────────────────┘

window: 33 tokens, SRC [74..106] / DST [63..95]; padding: SRC +0 / DST +1 newline tokens


The two token sequences (user turn onward). The **task span** whose keys / values the goal heads read is highlighted;
the last token of each prompt is the pivot (the connective). Only DST runs.

In [5]:
def show(tok, win, title, color):
    ids = tok['input_ids'][0].tolist()
    start = max(i for i, t in enumerate(ids) if t == ctx.mt.tokenizer.convert_tokens_to_ids('<|im_start|>') and i < min(win))
    txt = Text()
    for i in range(start, len(ids)):
        s = ctx.mt.tokenizer.decode([ids[i]])
        style = 'bold black on #fde68a' if i == len(ids) - 1 else f'bold {color} on #f1f5f9' if i in win else 'dim'
        txt.append(s, style=style)
    console.rule(title); console.print(txt)

show(g['src_tok'], set(g['src_w']), 'SRC prompt (k/v donor, C++)', 'cyan')
show(g['dst_tok'], set(g['dst_w']), 'DST prompt (the run we patch, Python)', 'magenta')

─────────────────────────────────────────────────────── SRC prompt (k/v donor, C++) ────────────────────────────────────────────────────────

<|im_start|>user
Write a C++ program that implements a function `func` which, given a list of strings `words`, computes:
(1) whether any word is empty
(2) the list of words in `words` that are shorter than four characters and start with the letter a
<|im_end|>
<|im_start|>assistant
<code>
auto func(std::vector<std::string> words){
    auto a = false;
    for(auto &a_w : words) if(a_w.empty()) a = true;
    std::vector<std::string> b;
    for (const auto& w : words) if (!w.empty() && w.size() < 4 &&

────────────────────────────────────────────────── DST prompt (the run we patch, Python) ───────────────────────────────────────────────────

<|im_start|>user
Write a Python program that implements a function `func` which, given a list of strings `words`, computes:

(1) whether any word is empty
(2) the list of words in `words` that are shorter than four characters and end with a digit
<|im_end|>
<|im_start|>assistant
<code>
def func(words: list[str]):
    a = any(len(w) == 0 for w in words)
    b = [w for w in words if len(w) < 4 and

## 2. Clean runs
`PP.clean_runs` greedily continues both prompts and takes a GoalScope readout at each pivot (the goal heads' write at
the connective, verbalized through the "What are you doing at this line of the code?" probe).

In [6]:
clean = PP.clean_runs(g, readout_heads=goal_heads, max_new=MAX_NEW, ctx=ctx)
for k in ('src', 'dst'):
    console.print(f'[bold]{k.upper()}[/] continuation of the connective: [green]{escape(clean[f"gen_{k}"].strip().splitlines()[0])}[/]')

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


pos=182 ' &&' => I'm checking if the string starts with 'a'.

pos=142 ' and' => I'm checking if the last character of a string is a digit.

SRC continuation of the connective: w[0] == 'a') b.push_back(w);

DST continuation of the connective: w[-1].isdigit()]

## 3. Patched run: KV transfer through the 250 goal heads (V × 4)
`PP.run_patched` = the same intervention as Demo 5 (SRC's k/v over the task span, read by the goal heads only, from
the pivot on) around a greedy decode, plus a GoalScope readout at the patched pivot. We also run the V × 1 arm and the
random-250 control (the paper ran the control without readouts; we add one for illustration).

In [7]:
arms = {
    'goal 250, V x4 (paper)':  dict(heads=goal_heads, v=V_SCALE),
    'goal 250, V x1':          dict(heads=goal_heads, v=1.0),
    'control 250, V x4':       dict(heads=ctrl_heads, v=V_SCALE),
}
pat = {name: PP.run_patched(g, goal_heads=a['heads'], readout_heads=goal_heads, v_scale=a['v'], max_new=MAX_NEW, ctx=ctx)
       for name, a in arms.items()}

pos=142 ' and' => I'm checking if the string starts with 'a'.

pos=142 ' and' => I'm checking if the line starts with 'a' and is exactly two characters long.

pos=142 ' and' => I'm checking if the last character of a string is a digit.

## 4. Judge
`PP.judge` is the batch eval's oracle: **causal** = does the patched continuation's *first* computation implement
SRC's sub-sub-goal `z`? (BOTH counts as a miss.) **readout** = are the clean readouts valid and did the patched
readout shift to `z`? For a Python destination the hier eval also **executes** the completed comprehension on a fixed
pool of words (`H.rule_verdict`): `src` = it selects exactly what `x and z` selects.

In [8]:
verd = {}
for name, p in pat.items():
    raw = dict(src_prompt=g['src_prompt'], dst_prompt=g['dst_prompt'], **clean, **p)
    verd[name] = PP.judge(inst, raw)
rules = {name: H.rule_verdict(inst, p['gen_patched']) for name, p in pat.items()}

first = lambda s: s.strip().splitlines()[0] if s.strip() else ''
lab = lambda l: {'src': '[bold cyan]z (SRC)[/]', 'dst': '[bold magenta]y (DST)[/]'}.get(l, f'[red]{l}[/]')
t = Table(title=f'SRC sub-sub-goal z = "{inst.src_cond.description}"   |   DST sub-sub-goal y = "{inst.dst_cond.description}"', show_lines=True)
t.add_column('run'); t.add_column('continuation of the connective', max_width=40); t.add_column('implements')
t.add_column('executed', max_width=10); t.add_column('GoalScope readout at the pivot', max_width=46); t.add_column('readout says')
rd0 = verd['goal 250, V x4 (paper)'][1]
t.add_row('clean SRC (C++)', escape(first(clean['gen_src'])), '-', '-', escape(clean['ro_src'].strip()), lab(rd0['source']))
t.add_row('clean DST', escape(first(clean['gen_dst'])), '-', H.rule_verdict(inst, clean['gen_dst'])['label'],
          escape(clean['ro_dst'].strip()), lab(rd0['dest']))
for name, p in pat.items():
    c, r = verd[name]
    hit = '[green]HIT[/]' if c['triggered'] else '[red]miss[/]'
    t.add_row(f'patched: {name}', escape(first(p['gen_patched'])), f"{lab(c['label'])} {hit}", rules[name]['label'],
              escape(p['ro_patched'].strip()), lab(r['patched']) + ('  [green]shifted[/]' if r['shifted'] else ''))
console.print(t)

                        SRC sub-sub-goal z = "start with the letter a"   |   DST sub-sub-goal y = "end with a digit"                        
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃                              ┃ continuation of the          ┃              ┃          ┃ GoalScope readout at the      ┃                  ┃
┃ run                          ┃ connective                   ┃ implements   ┃ executed ┃ pivot                         ┃ readout says     ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ clean SRC (C++)              │ w[0] == 'a') b.push_back(w); │ -            │ -        │ I'm checking if the string    │ z (SRC)          │
│                              │                              │              │          │ starts with 'a'.              │                  │
├──────────────────────────────┼──────────────────────────────┼──────────────┼──────────┼───────────────────────────────┼──────────────────┤
│ clean DST                    │ w[-1].isdigit()]             │ -            │ dst      │ I'm checking if the last      │ y (DST)          │
│                              │                              │              │          │ character of a string is a    │                  │
│                              │                              │              │          │ digit.                        │                  │
├──────────────────────────────┼──────────────────────────────┼──────────────┼──────────┼───────────────────────────────┼──────────────────┤
│ patched: goal 250, V x4      │ w.startswith('a')]           │ z (SRC) HIT  │ src      │ I'm checking if the string    │ z (SRC)  shifted │
│ (paper)                      │                              │              │          │ starts with 'a'.              │                  │
├──────────────────────────────┼──────────────────────────────┼──────────────┼──────────┼───────────────────────────────┼──────────────────┤
│ patched: goal 250, V x1      │ w.startswith('a')]           │ z (SRC) HIT  │ src      │ I'm checking if the line      │ z (SRC)  shifted │
│                              │                              │              │          │ starts with 'a' and is        │                  │
│                              │                              │              │          │ exactly two characters long.  │                  │
├──────────────────────────────┼──────────────────────────────┼──────────────┼──────────┼───────────────────────────────┼──────────────────┤
│ patched: control 250, V x4   │ w[-1].isdigit()]             │ y (DST) miss │ dst      │ I'm checking if the last      │ y (DST)          │
│                              │                              │              │          │ character of a string is a    │                  │
│                              │                              │              │          │ digit.                        │                  │
└──────────────────────────────┴──────────────────────────────┴──────────────┴──────────┴───────────────────────────────┴──────────────────┘

How to read it. Only the goal heads' read of the task span changed, and the Python destination now completes its
filter with the C++ source's second condition: the transplanted encoding carries the *sub-sub-goal*, not its C++
tokens. The executed comprehension selects exactly what `x and z` selects, and the GoalScope readout at the patched
pivot moves with it. The same transfer through 250 random heads leaves DST on `y`.

## 5. Cross-check against the reported run
The cached record of this instance (`results/diff/<model>/path_patch_hier/<run_tag>/<cell>/results.json`): same
greedy decodes, same verdicts (up to bf16 nondeterminism on other hardware). Then the reported rates for this cell and
pooled over all 32 cells (n = 1024).

In [9]:
R = os.path.join(env_utils.DEFAULT_RESULTS_DIR, 'diff', MODEL.split('/')[-1], 'path_patch_hier')
TAGS = {'goal 250, V x4 (paper)': 'I1-off__I2-off__I3-on-V4__win-span__span-list+gap__cppnote-sys__a1__th-goal_250__gh-goal_250__ro-goal_250',
        'goal 250, V x1':         'I1-off__I2-off__I3-on__win-span__span-list+gap__cppnote-sys__a1__th-goal_250__gh-goal_250__ro-goal_250',
        'control 250, V x4':      'I1-off__I2-off__I3-on-V4__win-span__span-list+gap__cppnote-sys__a1__th-control250_250__gh-control250_250__ro-off'}
cell = f'{PP._safe(DATATYPE)}__{CONNECTIVE}__{ORDER}__{SRC_LANG}-{DST_LANG}'
t = Table(title='this notebook vs. the reported run (same instance)')
t.add_column('run'); t.add_column('paper causal'); t.add_column('ours'); t.add_column('same patched gen?')
t.add_column(f'{cell} causal rate'); t.add_column('pooled causal (n)'); t.add_column('pooled readout shift')
for name, tag in TAGS.items():
    rec = next(r for r in json.load(open(os.path.join(R, tag, cell, 'results.json'))) if r['key'] == INST_KEY)
    meta = json.load(open(os.path.join(R, tag, cell, 'meta.json')))
    metas = [json.load(open(os.path.join(R, tag, d, 'meta.json'))) for d in sorted(os.listdir(os.path.join(R, tag)))
             if os.path.exists(os.path.join(R, tag, d, 'meta.json'))]
    n = sum(m['n_instances'] for m in metas)
    pc = sum(m['causal_trigger_rate'] * m['n_instances'] for m in metas) / n
    rs = [m for m in metas if m.get('readout_shift_rate') is not None]
    pr = (f"{sum(m['readout_shift_rate'] * m['n_instances'] for m in rs) / sum(m['n_instances'] for m in rs):.3f}" if rs else 'off')
    t.add_row(name, 'HIT' if rec['causal']['triggered'] else 'miss', 'HIT' if verd[name][0]['triggered'] else 'miss',
              str(rec['raw']['gen_patched'] == pat[name]['gen_patched']), f"{meta['causal_trigger_rate']:.3f}",
              f'{pc:.3f} ({n})', pr)
console.print(t)

                                             this notebook vs. the reported run (same instance)                                             
┏━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┓
┃                        ┃              ┃      ┃                   ┃ list_str__and__match__cpp… ┃                   ┃                      ┃
┃ run                    ┃ paper causal ┃ ours ┃ same patched gen? ┃ causal rate                ┃ pooled causal (n) ┃ pooled readout shift ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━┩
│ goal 250, V x4 (paper) │ HIT          │ HIT  │ True              │ 0.938                      │ 0.792 (1024)      │ 0.654                │
│ goal 250, V x1         │ HIT          │ HIT  │ True              │ 0.875                      │ 0.645 (1024)      │ 0.540                │
│ control 250, V x4      │ miss         │ miss │ True              │ 0.000                      │ 0.000 (1024)      │ off                  │
└────────────────────────┴──────────────┴──────┴───────────────────┴────────────────────────────┴───────────────────┴──────────────────────┘

## Where this goes in the paper / how to reproduce at scale
- **Commands**: `job_list/diff_eval/path_patch_hier_32b_listgap.json` (16 jobs = {list[int], list[str]} × 4 language
  pairs × {and, or}), e.g.
  ```
  python scripts/nrgs/kv_transfer_eval.py --setup hier --datatypes list[str] --lang-transfers cpp-py --connectives and --orders match diff --n-samples 32 --i3-v-scale 1 2 4
  ```
  and the random-head control, the 8 `--setup hier` jobs of `job_list/diff_eval/path_patch_control_32b_listgap.json`
  (`--goal-heads C250 --readout-heads C250 --no-readout`). Smoke: `--max-instances 2`. Records are resumable and land in
  `results/diff/Qwen2.5-Coder-32B-Instruct/path_patch_hier/<run_tag>/<dt>__<conn>__<order>__<src>-<dst>/`; point
  `RESULTS_DIR` in `env.yml` to a scratch dir for a re-run.
- **Paper numbers**: pooled n = 1024, causal 0.792, readout shift 0.654 (≈79%); random-250 control 0 / 1024.
- **Generator**: `notebooks/fig_n_table_generators/kv_transfer.ipynb` cell 9 (`kv_multihier_barcharts`, the right
  panel of `fig:kv-multi`).
- **Related**: Demo 5 (the same transfer one level up, the sub-goal) and Demo 5.2 (one more level up, the broader
  goal, with the next operation never written in the prompt).